In [1]:
from collections import Counter
import pandas as pd
from sklearn.cluster import SpectralClustering
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage2\525×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(525, 87)

In [2]:

def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage2\benchmark2.xlsx"
reported_structures = extract_numbers(file_path)

for j in range(30, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]

    gamma_values = np.logspace(-2, 2, 5)  # [0.001, 0.01, 0.1, 1, 10, 100, 1000]
    for i in range(3, 7):
        for gamma in gamma_values:
            SPC = SpectralClustering(n_clusters=i, affinity='rbf', gamma = gamma, assign_labels = 'kmeans', eigen_solver = 'arpack' , random_state=42)
            labels = SPC.fit_predict(pca_data)
            if len(np.unique(labels)) > 1:
                print(f"\nNumber of clusters k={i}:", f"\ngamma={gamma}:",f"\nPCA retained variance={j}:", f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
            else:
                pass

            data[f'cluster_{i}'] = labels

            dict = {}

            label_1 = []
            for num  in  reported_structures:
                label = labels[num-1]
                label_1.append(int(label))
            print(label_1)

            for cluster in range(i):

                cluster_samples = data[data[f'cluster_{i}'] == cluster]
                    # print(cluster_samples)

                total_in_cluster = len(cluster_samples)


                reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])
                dict[cluster] = (reported_in_cluster, total_in_cluster)
            for value in dict.values():
                if value[0] == 0:
                    for key, value in dict.items():

                        formatted_value = f"{value[0]}/{value[1]}"

                        print(f"Cluster {key}: {formatted_value}")
                        #print(f"Cluster {cluster}: {reported_in_cluster}/{total_in_cluster} "
                        #    f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


Number of clusters k=3: 
gamma=0.01: 
PCA retained variance=0.3: 
Silhouette score: 0.49 
Calinski-Harabasz score: 672.62
[0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 2, 2, 0]

Number of clusters k=3: 
gamma=0.1: 
PCA retained variance=0.3: 
Silhouette score: 0.49 
Calinski-Harabasz score: 670.55
[1, 1, 1, 2, 1, 2, 2, 2, 2, 1, 1, 2, 0, 0, 1]

Number of clusters k=3: 
gamma=1.0: 
PCA retained variance=0.3: 
Silhouette score: 0.49 
Calinski-Harabasz score: 676.84
[0, 0, 0, 2, 0, 2, 2, 2, 2, 0, 0, 2, 1, 1, 0]

Number of clusters k=3: 
gamma=10.0: 
PCA retained variance=0.3: 
Silhouette score: 0.48 
Calinski-Harabasz score: 669.24
[2, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 2, 2, 0]

Number of clusters k=3: 
gamma=100.0: 
PCA retained variance=0.3: 
Silhouette score: 0.36 
Calinski-Harabasz score: 354.08
[1, 0, 0, 2, 0, 2, 2, 2, 2, 0, 0, 2, 0, 0, 1]

Number of clusters k=4: 
gamma=0.01: 
PCA retained variance=0.3: 
Silhouette score: 0.45 
Calinski-Harabasz score: 693.18
[0, 0, 0, 2, 0, 3, 2, 3, 3, 0, 0,

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited at iteration 1061 with accuracies 
[5.33255630e-15 8.08409869e-06 2.65174855e-05 2.17099399e-05
 7.81910709e-06]
not reaching the requested tolerance 7.82310962677002e-06.
Use iteration 1061 instead with accuracy 
1.2826126247568703e-05.

  _, diffusion_map = lobpcg(
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited postprocessing with accuracies 
[4.11570639e-15 8.08416228e-06 2.65175633e-05 2.17098209e-05
 7.81910792e-06]
not reaching the requested tolerance 7.82310962677002e-06.
  _, diffusion_map = lobpcg(



Number of clusters k=4: 
gamma=100.0: 
PCA retained variance=0.35: 
Silhouette score: 0.12 
Calinski-Harabasz score: 136.30
[2, 2, 2, 0, 2, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2]
Cluster 0: 6/210
Cluster 1: 0/2
Cluster 2: 9/308
Cluster 3: 0/5
Cluster 0: 6/210
Cluster 1: 0/2
Cluster 2: 9/308
Cluster 3: 0/5

Number of clusters k=5: 
gamma=0.01: 
PCA retained variance=0.35: 
Silhouette score: 0.32 
Calinski-Harabasz score: 315.80
[2, 0, 2, 3, 2, 1, 3, 1, 1, 0, 2, 3, 4, 2, 2]

Number of clusters k=5: 
gamma=0.1: 
PCA retained variance=0.35: 
Silhouette score: 0.32 
Calinski-Harabasz score: 327.51
[4, 3, 4, 1, 4, 2, 1, 2, 2, 3, 4, 1, 0, 4, 4]

Number of clusters k=5: 
gamma=1.0: 
PCA retained variance=0.35: 
Silhouette score: 0.31 
Calinski-Harabasz score: 322.75
[3, 1, 3, 4, 3, 0, 4, 0, 0, 1, 3, 4, 2, 3, 3]

Number of clusters k=5: 
gamma=10.0: 
PCA retained variance=0.35: 
Silhouette score: 0.29 
Calinski-Harabasz score: 284.09
[2, 2, 2, 1, 2, 1, 1, 0, 1, 3, 2, 1, 4, 2, 2]

Number of clusters k=5

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited at iteration 1061 with accuracies 
[5.33255630e-15 8.08409869e-06 2.65174855e-05 2.17099399e-05
 7.81910709e-06]
not reaching the requested tolerance 7.82310962677002e-06.
Use iteration 1061 instead with accuracy 
1.2826126247568703e-05.

  _, diffusion_map = lobpcg(
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited postprocessing with accuracies 
[4.11570639e-15 8.08416228e-06 2.65175633e-05 2.17098209e-05
 7.81910792e-06]
not reaching the requested tolerance 7.82310962677002e-06.
  _, diffusion_map = lobpcg(



Number of clusters k=4: 
gamma=100.0: 
PCA retained variance=0.4: 
Silhouette score: 0.12 
Calinski-Harabasz score: 136.30
[2, 2, 2, 0, 2, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2]
Cluster 0: 6/210
Cluster 1: 0/2
Cluster 2: 9/308
Cluster 3: 0/5
Cluster 0: 6/210
Cluster 1: 0/2
Cluster 2: 9/308
Cluster 3: 0/5

Number of clusters k=5: 
gamma=0.01: 
PCA retained variance=0.4: 
Silhouette score: 0.32 
Calinski-Harabasz score: 315.80
[2, 0, 2, 3, 2, 1, 3, 1, 1, 0, 2, 3, 4, 2, 2]

Number of clusters k=5: 
gamma=0.1: 
PCA retained variance=0.4: 
Silhouette score: 0.32 
Calinski-Harabasz score: 327.51
[4, 3, 4, 1, 4, 2, 1, 2, 2, 3, 4, 1, 0, 4, 4]

Number of clusters k=5: 
gamma=1.0: 
PCA retained variance=0.4: 
Silhouette score: 0.31 
Calinski-Harabasz score: 322.75
[3, 1, 3, 4, 3, 0, 4, 0, 0, 1, 3, 4, 2, 3, 3]

Number of clusters k=5: 
gamma=10.0: 
PCA retained variance=0.4: 
Silhouette score: 0.29 
Calinski-Harabasz score: 284.09
[2, 2, 2, 1, 2, 1, 1, 0, 1, 3, 2, 1, 4, 2, 2]

Number of clusters k=5: 
ga

...

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited at iteration 175 with accuracies 
[1.14790380e-14 9.30052051e-06 3.56268996e-06 8.96207315e-06
 7.46779614e-06]
not reaching the requested tolerance 7.82310962677002e-06.
Use iteration 151 instead with accuracy 
5.542110579072768e-06.

  _, diffusion_map = lobpcg(
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited postprocessing with accuracies 
[7.05423724e-15 8.55103619e-06 3.14518070e-06 7.28559544e-06
 8.72874072e-06]
not reaching the requested tolerance 7.82310962677002e-06.
  _, diffusion_map = lobpcg(
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_met


Number of clusters k=4: 
gamma=100.0: 
PCA retained variance=0.95: 
Silhouette score: 0.32 
Calinski-Harabasz score: 3.17
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0

Number of clusters k=5: 
gamma=0.01: 
PCA retained variance=0.95: 
Silhouette score: 0.10 
Calinski-Harabasz score: 56.41
[0, 4, 0, 2, 0, 2, 2, 3, 2, 4, 0, 2, 1, 0, 0]

Number of clusters k=5: 
gamma=0.1: 
PCA retained variance=0.95: 
Silhouette score: 0.10 
Calinski-Harabasz score: 57.55
[0, 2, 0, 1, 0, 1, 1, 4, 1, 2, 0, 1, 3, 0, 0]

Number of clusters k=5: 
gamma=1.0: 
PCA retained variance=0.95: 
Silhouette score: 0.19 
Calinski-Harabasz score: 38.61
[0, 0, 0, 3, 0, 3, 3, 3, 3, 0, 0, 3, 0, 0, 0]
Cluster 0: 9/309
Cluster 1: 0/2
Cluster 2: 0/3
Cluster 3: 6/210
Cluster 4: 0/1
Cluster 0: 9/309
Cluster 1: 0/2
Cluster 2: 0/3
Cluster 3: 

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



Number of clusters k=5: 
gamma=100.0: 
PCA retained variance=0.95: 
Silhouette score: 0.32 
Calinski-Harabasz score: 3.17
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 4: 0/0
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 4: 0/0
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 4: 0/0
Cluster 0: 15/524
Cluster 1: 0/1
Cluster 2: 0/0
Cluster 3: 0/0
Cluster 4: 0/0

Number of clusters k=6: 
gamma=0.01: 
PCA retained variance=0.95: 
Silhouette score: 0.10 
Calinski-Harabasz score: 50.46
[2, 2, 2, 4, 5, 4, 4, 4, 5, 3, 5, 4, 2, 0, 0]
Cluster 0: 2/119
Cluster 1: 0/47
Cluster 2: 4/85
Cluster 3: 1/70
Cluster 4: 5/134
Cluster 5: 3/70

Number of clusters k=6: 
gamma=0.1: 
PCA retained variance=0.95: 
Silhouette score: 0.10 
Calinski-Harabasz score: 50.28
[4, 4, 4, 1, 5, 1, 1, 1, 5, 3, 5, 1, 0, 2, 4]

Number of clusters k=6: 
gamma=1.0: 
PCA retained variance=0.95: 
Silhouette score:

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:453: UserWarning: Exited postprocessing with accuracies 
[3.51349124e-15 8.32963724e-07 2.40863368e-06 3.59409351e-06
 4.39751390e-06 4.70172860e-06 7.89767007e-06]
not reaching the requested tolerance 7.82310962677002e-06.
  _, diffusion_map = lobpcg(
c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [5]:
pca = PCA(n_components= 0.5)
pca_data = pca.fit_transform(X)
print(pca_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)


i = 6
SPC = SpectralClustering(n_clusters=i, affinity='rbf', gamma = 0.1, assign_labels = 'kmeans', eigen_solver = 'arpack' , random_state=42)
labels = SPC.fit_predict(pca_data)

df = pd.DataFrame(labels)
df.to_excel("SPC_result_1.xlsx", index=False)

data[f'cluster_{i}'] = labels

print(f"\nNumber of clusters k={i}:")
print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
label_1 = []
for num  in  reported_structures:
        label = labels[num-1]
        label_1.append(int(label))
print(label_1)


for cluster in range(i):

    cluster_samples = data[data[f'cluster_{i}'] == cluster]
            # print(cluster_samples)

    total_in_cluster = len(cluster_samples)
    # print(total_in_cluster)

    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])

    print(f"Cluster {cluster}: {reported_in_cluster}/{total_in_cluster} "
                f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


(525, 5)

Number of clusters k=6:

Silhouette score: 0.20 
Calinski-Harabasz score: 131.91
[4, 5, 4, 1, 3, 1, 1, 1, 3, 2, 3, 1, 5, 4, 4]
Cluster 0: 0/86 (Reported ratio: 0.00%)
Cluster 1: 5/126 (Reported ratio: 3.97%)
Cluster 2: 1/76 (Reported ratio: 1.32%)
Cluster 3: 3/64 (Reported ratio: 4.69%)
Cluster 4: 4/105 (Reported ratio: 3.81%)
Cluster 5: 2/68 (Reported ratio: 2.94%)


In [6]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook


wb = Workbook()
ws = wb.active


for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)


red_bold_font = Font(color="FF0000", bold=True)


for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]


    values = [0 if v is None else v for v in values]


    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]


    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font


output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")

The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
